In [1]:
import requests
import pandas as pd
import numpy as np
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
import requests
import time
import pandas as pd

url = "https://api.binance.com/api/v3/klines"
symbol = "BTCUSDT"
interval = "4h"
limit = 1000
all_data = []
end_time = None

for i in range(5):  # 5 × 1000 = 5000 کندل
    params = {
        "symbol": symbol,
        "interval": interval,
        "limit": limit
    }
    if end_time:
        params["endTime"] = end_time

    response = requests.get(url, params=params)
    data = response.json()

    if not data:
        break

    all_data = data + all_data

    # زمان قدیمی‌ترین کندل این دسته، منهای یک میلی‌ثانیه
    end_time = data[0][0] - 1

    print(f"تا الان {len(all_data)} کندل جمع شد...")
    time.sleep(0.5)

تا الان 1000 کندل جمع شد...
تا الان 2000 کندل جمع شد...
تا الان 3000 کندل جمع شد...
تا الان 4000 کندل جمع شد...
تا الان 5000 کندل جمع شد...


In [3]:
df = pd.DataFrame(all_data, columns=[
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore"
])

df

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,1710417600000,72884.01000000,73047.16000000,70543.64000000,70844.01000000,18557.28967000,1710431999999,1333661629.90618070,774473,9250.88373000,664898688.71421630,0
1,1710432000000,70844.00000000,71458.00000000,68555.00000000,69415.09000000,22921.42581000,1710446399999,1608082774.61199250,852708,11114.15832000,780380292.01271780,0
2,1710446400000,69415.09000000,71709.90000000,69303.75000000,71388.94000000,10667.10021000,1710460799999,756262443.72533290,401598,5791.57865000,410599503.48412920,0
3,1710460800000,71388.94000000,72419.71000000,66699.00000000,68224.01000000,22422.85781000,1710475199999,1557312351.48532790,898751,10358.66957000,720253486.44285230,0
4,1710475200000,68224.01000000,68861.10000000,66904.98000000,68448.29000000,17229.30114000,1710489599999,1170918348.66922130,605870,8502.42870000,577978158.09779330,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,1782345600000,61078.00000000,61163.16000000,60684.94000000,60883.65000000,2441.96666000,1782359999999,148617037.71762240,488365,1286.89158000,78325445.13785370,0
4996,1782360000000,60883.66000000,61962.40000000,60792.00000000,61911.04000000,3236.44626000,1782374399999,199336748.64757960,585612,1792.57753000,110375492.28432150,0
4997,1782374400000,61911.03000000,61920.00000000,61066.00000000,61282.01000000,1957.14848000,1782388799999,120376027.91725560,397068,927.34829000,57042676.00598120,0
4998,1782388800000,61282.00000000,61761.35000000,58115.01000000,59557.99000000,15979.29296000,1782403199999,950943210.77604520,2405299,7129.72909000,424485511.14668750,0


In [4]:
df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

df = df.drop(columns=["ignore"])

df

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
0,2024-03-14 12:00:00,72884.01000000,73047.16000000,70543.64000000,70844.01000000,18557.28967000,2024-03-14 15:59:59.999,1333661629.90618070,774473,9250.88373000,664898688.71421630
1,2024-03-14 16:00:00,70844.00000000,71458.00000000,68555.00000000,69415.09000000,22921.42581000,2024-03-14 19:59:59.999,1608082774.61199250,852708,11114.15832000,780380292.01271780
2,2024-03-14 20:00:00,69415.09000000,71709.90000000,69303.75000000,71388.94000000,10667.10021000,2024-03-14 23:59:59.999,756262443.72533290,401598,5791.57865000,410599503.48412920
3,2024-03-15 00:00:00,71388.94000000,72419.71000000,66699.00000000,68224.01000000,22422.85781000,2024-03-15 03:59:59.999,1557312351.48532790,898751,10358.66957000,720253486.44285230
4,2024-03-15 04:00:00,68224.01000000,68861.10000000,66904.98000000,68448.29000000,17229.30114000,2024-03-15 07:59:59.999,1170918348.66922130,605870,8502.42870000,577978158.09779330
...,...,...,...,...,...,...,...,...,...,...,...
4995,2026-06-25 00:00:00,61078.00000000,61163.16000000,60684.94000000,60883.65000000,2441.96666000,2026-06-25 03:59:59.999,148617037.71762240,488365,1286.89158000,78325445.13785370
4996,2026-06-25 04:00:00,60883.66000000,61962.40000000,60792.00000000,61911.04000000,3236.44626000,2026-06-25 07:59:59.999,199336748.64757960,585612,1792.57753000,110375492.28432150
4997,2026-06-25 08:00:00,61911.03000000,61920.00000000,61066.00000000,61282.01000000,1957.14848000,2026-06-25 11:59:59.999,120376027.91725560,397068,927.34829000,57042676.00598120
4998,2026-06-25 12:00:00,61282.00000000,61761.35000000,58115.01000000,59557.99000000,15979.29296000,2026-06-25 15:59:59.999,950943210.77604520,2405299,7129.72909000,424485511.14668750


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   open_time               5000 non-null   datetime64[ms]
 1   open                    5000 non-null   str           
 2   high                    5000 non-null   str           
 3   low                     5000 non-null   str           
 4   close                   5000 non-null   str           
 5   volume                  5000 non-null   str           
 6   close_time              5000 non-null   datetime64[ms]
 7   quote_asset_volume      5000 non-null   str           
 8   number_of_trades        5000 non-null   int64         
 9   taker_buy_base_volume   5000 non-null   str           
 10  taker_buy_quote_volume  5000 non-null   str           
dtypes: datetime64[ms](2), int64(1), str(8)
memory usage: 429.8 KB


In [6]:
cols = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_asset_volume",
    "taker_buy_base_volume",
    "taker_buy_quote_volume"
]

for col in cols:
    df[col] = df[col].astype(float)

In [8]:
df["future_return"] = (df["close"].shift(-7) / df["close"] - 1)

def label(x):
    if x > 0.015:
        return 1
    elif x < -0.015:
        return -1

df["target"] = df["future_return"].apply(label)

In [9]:
df = df.dropna()
df

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,future_return,target
0,2024-03-14 12:00:00,72884.01,73047.16,70543.64,70844.01,18557.28967,2024-03-14 15:59:59.999,1.333662e+09,774473,9250.88373,6.648987e+08,-0.026537,-1.0
2,2024-03-14 20:00:00,69415.09,71709.90,69303.75,71388.94,10667.10021,2024-03-14 23:59:59.999,7.562624e+08,401598,5791.57865,4.105995e+08,-0.030661,-1.0
3,2024-03-15 00:00:00,71388.94,72419.71,66699.00,68224.01,22422.85781,2024-03-15 03:59:59.999,1.557312e+09,898751,10358.66957,7.202535e+08,0.017713,1.0
6,2024-03-15 12:00:00,67757.75,68746.69,67424.79,68157.07,12481.72359,2024-03-15 15:59:59.999,8.502483e+08,565151,6341.27590,4.320175e+08,-0.018275,-1.0
7,2024-03-15 16:00:00,68157.07,70650.00,67520.01,68964.00,17320.34953,2024-03-15 19:59:59.999,1.194156e+09,596512,8711.96839,6.007204e+08,-0.053120,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4986,2026-06-23 12:00:00,62507.05,62855.98,61960.00,62487.79,4100.47989,2026-06-23 15:59:59.999,2.558907e+08,946398,1894.29980,1.182245e+08,-0.040480,-1.0
4987,2026-06-23 16:00:00,62487.79,62846.00,62104.70,62388.49,2461.76341,2026-06-23 19:59:59.999,1.537688e+08,580044,1214.83572,7.592024e+07,-0.021005,-1.0
4988,2026-06-23 20:00:00,62388.49,62799.99,62380.25,62734.57,1483.69383,2026-06-23 23:59:59.999,9.283524e+07,369212,941.14695,5.888592e+07,-0.029504,-1.0
4990,2026-06-24 04:00:00,62729.78,63073.44,62525.49,62657.99,1825.36336,2026-06-24 07:59:59.999,1.145388e+08,343629,869.10988,5.453939e+07,-0.021960,-1.0


In [10]:
print(df["target"].value_counts())

target
 1.0    1183
-1.0    1181
Name: count, dtype: int64


## 📊 Feature Engineering

In this project, we create technical features from raw OHLCV data to help the model understand market behavior.

### 1. Return
Measures the percentage change in price between consecutive days.

- Formula:  
  `return = (close_t - close_{t-1}) / close_{t-1}`

- Purpose:  
  Captures short-term price movement direction.

---

### 2. Simple Moving Average (SMA)

Average price over a fixed window (e.g., 20 days).

- Formula:  
  `SMA20 = mean(close over last 20 days)`

- Purpose:  
  Identifies overall market trend (uptrend/downtrend).

---

### 3. Volatility

Measures how much price fluctuates over time.

- Formula:  
  `volatility = std(return over window)`

- Purpose:  
  Indicates market risk and instability.

---

## 🎯 Why these features?

Raw price data alone is noisy.  
These features help the model understand:

- Trend direction
- Market momentum
- Risk level

In [12]:
df["return_3"] = df["close"].pct_change(3)
df["return_7"] = df["close"].pct_change(7)

In [13]:
df["sma7"] = df["close"].rolling(7).mean()
df["sma20"] = df["close"].rolling(20).mean()
df["sma50"] = df["close"].rolling(50).mean()

In [14]:
df["hl_range"] = df["high"] - df["low"]
df["volatility_14"] = df["hl_range"].rolling(14).mean()

In [15]:
df["price_sma_ratio"] = df["close"] / df["sma20"]

In [16]:
delta = df["close"].diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()

rs = gain / loss
df["rsi"] = 100 - (100 / (1 + rs))

In [17]:
df["bb_mid"] = df["close"].rolling(20).mean()
df["bb_std"] = df["close"].rolling(20).std()

df["bb_upper"] = df["bb_mid"] + 2 * df["bb_std"]
df["bb_lower"] = df["bb_mid"] - 2 * df["bb_std"]

In [18]:
# نسبت حجم فعلی به میانگین حجم ۲۰ روز اخیر
df["volume_ratio"] = df["volume"] / df["volume"].rolling(20).mean()

# مومنتوم: تغییر قیمت در ۱۴ روز اخیر
df["momentum_14"] = df["close"].pct_change(14)

# فاصله قیمت از باند بولینگر (موقعیت نسبی قیمت)
df["bb_position"] = (df["close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])

In [19]:
DF = df.dropna()

In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 2364 entries, 0 to 4991
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   open_time               2364 non-null   datetime64[ms]
 1   open                    2364 non-null   float64       
 2   high                    2364 non-null   float64       
 3   low                     2364 non-null   float64       
 4   close                   2364 non-null   float64       
 5   volume                  2364 non-null   float64       
 6   close_time              2364 non-null   datetime64[ms]
 7   quote_asset_volume      2364 non-null   float64       
 8   number_of_trades        2364 non-null   int64         
 9   taker_buy_base_volume   2364 non-null   float64       
 10  taker_buy_quote_volume  2364 non-null   float64       
 11  future_return           2364 non-null   float64       
 12  target                  2364 non-null   float64       
 13  volu

In [21]:
split_index = int(len(df) * 0.8)

train = df[:split_index]
test = df[split_index:]

In [22]:
features = [
    "return_3",
    "return_7",
    "volatility_14",
    "price_sma_ratio",
    "rsi",
    "volume_ratio",
    "momentum_14",
    "bb_position"
]
X_train = train[features]
y_train = train["target"]

X_test = test[features]
y_test = test["target"]

# RandomForest

In [33]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42
)

In [34]:
rf_model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [35]:
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.5539112050739958
[[135 133]
 [ 78 127]]
              precision    recall  f1-score   support

        -1.0       0.63      0.50      0.56       268
         1.0       0.49      0.62      0.55       205

    accuracy                           0.55       473
   macro avg       0.56      0.56      0.55       473
weighted avg       0.57      0.55      0.55       473



# XGBoost

In [26]:
y_train_xgb = y_train.replace({
    -1: 0,
     1: 1
})

y_test_xgb = y_test.replace({
    -1: 0,
     1: 1
})

In [27]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

In [28]:
xgb_model.fit(
    X_train,
    y_train_xgb
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [29]:
y_pred = xgb_model.predict(X_test)

In [30]:
print(
    accuracy_score(y_test_xgb,y_pred)
)

print(
    confusion_matrix(y_test_xgb,y_pred)
)

print(
    classification_report(y_test_xgb,y_pred)
)

0.5412262156448203
[[137 131]
 [ 86 119]]
              precision    recall  f1-score   support

         0.0       0.61      0.51      0.56       268
         1.0       0.48      0.58      0.52       205

    accuracy                           0.54       473
   macro avg       0.55      0.55      0.54       473
weighted avg       0.55      0.54      0.54       473



# save model

In [39]:
import joblib

# ذخیره مدل آموزش‌دیده در یک فایل
joblib.dump(rf_model, "model/rf_model.pkl")

print("مدل ذخیره شد.")

مدل ذخیره شد.


In [40]:
import json

# ذخیره اسم فیچرها به ترتیب، برای استفاده در پیش‌بینی‌های آینده
with open("model/features.json", "w") as f:
    json.dump(features, f)

print("لیست فیچرها هم ذخیره شد.")

لیست فیچرها هم ذخیره شد.
